# 独立 XGB/LGB 三资产配置对照

冻结的来源、时钟、标签、模型、费用、双路径与门槛见 `plan.json`。该卡从原生 Qlib Recorder 读取已验证全路径数据；与旧 v2 3bp 简化成本结果分开。


In [1]:
import xgboost  # Must precede Qlib/PyTorch imports on this macOS runtime.
import contextlib
import json
from pathlib import Path
from src.workflow.independent_tree_allocation import run
from src.workflow.independent_tree_regimes import finalize, constraint_bridge
root = Path.cwd().resolve()
card = root / "research/independent_tree_allocation_20261027"
plan = json.loads((card / "plan.json").read_text())
with open("/tmp/independent_tree_allocation_20261027_run.log", "w") as log, contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
    summary = run(root=root, card=card, plan=plan)
with open("/tmp/independent_tree_allocation_20261027_regimes.log", "w") as log, contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
    summary = finalize(card=card)
    bridge = constraint_bridge(card=card)
print(summary["decision"], summary["regime_evidence"], bridge["status"])


not_promoted_0bp_gate_failed {'status': 'passed', 'annual_rows': 24, 'quarter_rows': 96, 'window_rows': 4914, 'windows_per_case': 819, 'quarters_per_case': 16} diagnostic_post_model_mapping


## 中文结论

冻结同一训练标签和输入，分别拟合 XGB/LGB；每个模型共用预测，比较放开限制与加回限制的自身完整账户。0bp仍收Tiered费用。

|case|return_pct|relative_wealth|max_drawdown_pct|positive_126d_fraction|average_soxl_pct|
|---|---|---|---|---|---|
|xgb_unrestricted_0|813.3694|1.8726|-79.2429|0.6178|63.6080|
|xgb_constrained_0|21.6432|0.2494|-12.4844|0.1648|2.5224|
|lgbm_unrestricted_0|279.5731|0.7782|-65.6391|0.5226|55.0383|
|lgbm_constrained_0|16.8938|0.2397|-10.0059|0.1832|2.3827|
|reference_half_soxl_cash_0|564.0096|1.3614|-61.9585|0.6313|50.0716|
|reference_full_soxl_0|1333.4178|2.9388|-88.7954|0.6325|99.9011|


决定：`not_promoted_0bp_gate_failed`。2023—2026 是反复研究历史，不构成独立前向；旧 v2 的简化3bp结果不与本账户混用。

## 区间与风险解读

完整年度与两端部分年度见 `annual_metrics.csv`；每组 16 个季度见 `quarter_metrics.csv`，全部 819 个连续126日窗口存于 Qlib Recorder `analysis/window_detail`，极值见 `window_summary.json`。

XGB放开版虽费用后超过SOXX，但平均名义 beta 约1.91、SOXL约63.6%，回撤约79%、CVaR5约-11.75%、窗口胜出61.8%；同一历史的固定50% SOXL+现金参考也超过SOXX，故不能把超额全归于择时。LGB放开版期末相对财富约0.778；加回旧限制后两模型现金均超过92%，显示限制在这组固定动作预测上显著压低参与。

训练标签1060行中，现金500、SOXL552、SOXX仅8，学习问题对SOXX选择支持很弱；共同现金起点的一步标签也不等于自身持仓路径上的长期净价值。这里的监测回撤来自可用09:45及收盘点位，未认证完整分钟极值，但候选已在这些可用点位上失败。所有候选0bp至少一项硬门失败，固定版本关闭，不进入2/5bp、不改现役release；不能将本卡直接当作精确f696+b472的同钟配对改进或独立前向证据。

### 限制传递

XGB：944日原始动作中SOXL 601日、SOXX 0日、现金 343日；受限账户自身回撤触发4%止损 864日，受限目标现金均值 92.89%。

LGBM：944日原始动作中SOXL 520日、SOXX 0日、现金 424日；受限账户自身回撤触发4%止损 909日，受限目标现金均值 93.29%。

这是仓位上限、现金下限和止损合并后的映射及两条自身账户路径；未单独识别每项限制的因果贡献。

## 2026-10-28 方法审计补充

方法纠正：一步共同现金起点的纯动作最优标签只有8/1060日选SOXX，硬argmax并不是连续三资产配置；受限组同时施加30% SOXL上限、15%现金底线和4%止损，不能隔离哪一条限制压低收益。冻结账户数字保持原口径。 [证据与完整作用域](../stage1_methodology_audit_20261028/conclusion.md)。
